# EDA und Datenvorbereitung

Ziel dieses Notebooks ist es, die Data-Collection-Outputs zu verstehen, erste Qualitätschecks durchzuführen und die Datasets für die Analyse vorzubereiten

In [106]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

In [107]:
DATA_DIR = Path("../../data")
INTERIM_DIR = DATA_DIR / "interim"

In [108]:
COSTS_FILE = INTERIM_DIR / "gesundheitskosten_2011_2026.csv"

df_costs = pd.read_csv(COSTS_FILE)

print(pd.DataFrame({
        "rows": [len(df_costs)],
        "columns": [df_costs.shape[1]],
        "duplicate_rows": [df_costs.duplicated().sum()],
        "missing_cells": [df_costs.isna().sum().sum()],
}))
empty_columns = df_costs.columns[df_costs.isna().all()].tolist()
print(empty_columns)

Die Gesundheitskosten enthalten 44’232 fehlende Werte. Diese entstehen  durch sechs optionale Metadatenspalten die leer sind. Die analyse-relevanten Spalten wie Jahr, Kanton, Alter, Beobachtungswert, Multiplikator und Status enthalten keine fehlenden Werte.

Es sind ebenfalls viele Spalten vorhanden, welche für die Analyse später nicht relevant sind.

In [109]:
cost_columns = [
    "TIME_PERIOD",
    "CANTON",
    "Swiss cantons",
    "AGE",
    "Age groups",
    "OBS_VALUE",
    "MULT",
    "OBS_STATUS",
    "Code list for Observation Status",
]

df_costs_processed = df_costs[cost_columns].copy()
df_costs_processed.head(30)

,TIME_PERIOD,CANTON,Swiss cantons,AGE,Age groups,OBS_VALUE,MULT,OBS_STATUS,Code list for Observation Status
0,2011,_T,Total,_T,Total,64234.604,6,A,Normal value
1,2012,_T,Total,_T,Total,66521.482,6,A,Normal value
2,2013,_T,Total,_T,Total,69352.477,6,A,Normal value
3,2014,_T,Total,_T,Total,71055.556,6,A,Normal value
4,2015,_T,Total,_T,Total,73345.969,6,A,Normal value
5,2016,_T,Total,_T,Total,75935.877,6,A,Normal value
6,2017,_T,Total,_T,Total,77866.583,6,A,Normal value
7,2018,_T,Total,_T,Total,79171.643,6,A,Normal value
8,2019,_T,Total,_T,Total,81686.113,6,A,Normal value
9,2020,_T,Total,_T,Total,83522.010,6,A,Normal value


In [110]:
pd.DataFrame({
        "year_min": [df_costs_processed["TIME_PERIOD"].min()],
        "year_max": [df_costs_processed["TIME_PERIOD"].max()],
        "n_years": [df_costs_processed["TIME_PERIOD"].nunique()],
        "n_cantons": [df_costs_processed["Swiss cantons"].nunique()],
        "n_age_groups": [df_costs_processed["Age groups"].nunique()],
        "n_status": [df_costs_processed["OBS_STATUS"].nunique()],
        "n_multipliers": [df_costs_processed["MULT"].nunique()],
})

,year_min,year_max,n_years,n_cantons,n_age_groups,n_status,n_multipliers
0,2011,2024,14,27,21,3,1


In [111]:
print(f'Status: \n {df_costs_processed["OBS_STATUS"].unique()}')
print(f'Kantone: \n {df_costs_processed["Swiss cantons"].unique()}')

In [112]:
pd.crosstab(
    df_costs_processed["TIME_PERIOD"],
    df_costs_processed["OBS_STATUS"]
)

OBS_STATUS,A,E,P
TIME_PERIOD,,,
2011,567,0,0
2012,567,0,0
2013,567,0,0
2014,567,0,0
2015,567,0,0
2016,567,0,0
2017,567,0,0
2018,567,0,0
2019,567,0,0


Für den Status haben 3 verschiedene Werte:

- A: Normaler Wert
- E: Geschätzter Wert
- P: Provisorischer Wert

Das Jahr 2023 beinhaltet nur provisorische Werte für alle Zeilen.
Das Jahr 2024 beinhaltet insgesamt nur einen geschätzten Wert.

Die Spalte Kantone beinhaltet alle 26 Kantone und das Total für die Schweiz

Um den absolut Wert für die Kosten zu erhalten müssen wird eine zusätzliche Spalte einfügen

In [113]:
df_costs_processed["costs_chf"] = df_costs["OBS_VALUE"] * (10 ** df_costs["MULT"])
df_costs_processed.head()

,TIME_PERIOD,CANTON,Swiss cantons,AGE,Age groups,OBS_VALUE,MULT,OBS_STATUS,Code list for Observation Status,costs_chf
0,2011,_T,Total,_T,Total,64234.604,6,A,Normal value,6.423460e+10
1,2012,_T,Total,_T,Total,66521.482,6,A,Normal value,6.652148e+10
2,2013,_T,Total,_T,Total,69352.477,6,A,Normal value,6.935248e+10
3,2014,_T,Total,_T,Total,71055.556,6,A,Normal value,7.105556e+10
4,2015,_T,Total,_T,Total,73345.969,6,A,Normal value,7.334597e+10


In [114]:
df_costs_total_ch = df_costs_processed[
    (df_costs_processed["Swiss cantons"] == "Total") &
    (df_costs_processed["Age groups"] == "Total")
].copy()

px.line(
    df_costs_total_ch,
    x="TIME_PERIOD",
    y="costs_chf",
    markers=True,
    title="Gesundheitskosten Schweiz, Total",
    labels={"TIME_PERIOD": "Jahr", "costs_chf": "Kosten in CHF"},
)

In [115]:
df_costs_canton_total = df_costs_processed[
    (df_costs_processed["Age groups"] == "Total") &
    (df_costs_processed["Swiss cantons"] != "Total")
].copy()

latest_canton_year = int(df_costs_canton_total["TIME_PERIOD"].max())

df_costs_canton_latest = (
    df_costs_canton_total[df_costs_canton_total["TIME_PERIOD"] == latest_canton_year]
    .sort_values("costs_chf", ascending=False)
)

px.bar(
    df_costs_canton_latest.head(10),
    x="Swiss cantons",
    y="costs_chf",
    title=f"Top 10 Kantone nach Gesundheitskosten {latest_canton_year}",
)

Im Plot für die Top 10 Kantone ist zu sehen, dass die Bevölkerungsgrösse eine Rolle spielt.

Für die spätere Analyse könnten die pro Kopf Kosten einen besseren Einblick geben

In [116]:
df_costs_age_ch = df_costs_processed[
    (df_costs_processed["Swiss cantons"] == "Total") &
    (df_costs_processed["Age groups"] != "Total") &
    (df_costs_processed["TIME_PERIOD"] == 2023)
].copy()

df_costs_age_ch["cost_share"] = (
    df_costs_age_ch["costs_chf"] / df_costs_age_ch["costs_chf"].sum()
)

px.bar(
    df_costs_age_ch,
    x="Age groups",
    y="cost_share",
    title="Kostenanteil nach Altersgruppe Schweiz 2023",
)

In [145]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

df_costs_processed.to_csv(PROCESSED_DIR / "gesundheitskosten.csv", index=False)

# 2. Bevölkerung

In [118]:
POPULATION_FILE = INTERIM_DIR / "bevoelkerung_2011_2026.csv"

df_population = pd.read_csv(POPULATION_FILE)
pd.DataFrame({
        "rows": [len(df_population)],
        "columns": [df_population.shape[1]],
        "duplicate_rows": [df_population.duplicated().sum()],
        "missing_cells": [df_population.isna().sum().sum()],
})

,rows,columns,duplicate_rows,missing_cells
0,39984,6,0,0


In [119]:
df_population.head()

,Jahr,Kanton,Staatsangehörigkeit (Kategorie),Geschlecht,Alter,Bestand am 31. Dezember
0,2011,Schweiz,Staatsangehörigkeit (Kategorie) - Total,Geschlecht - Total,Alter - Total,7954662
1,2011,Schweiz,Staatsangehörigkeit (Kategorie) - Total,Geschlecht - Total,0 Jahre,78426
2,2011,Schweiz,Staatsangehörigkeit (Kategorie) - Total,Geschlecht - Total,1 Jahr,81663
3,2011,Schweiz,Staatsangehörigkeit (Kategorie) - Total,Geschlecht - Total,2 Jahre,80196
4,2011,Schweiz,Staatsangehörigkeit (Kategorie) - Total,Geschlecht - Total,3 Jahre,79522


In [120]:
print(df_population["Staatsangehörigkeit (Kategorie)"].unique())
print(df_population["Geschlecht"].unique())

<StringArray>
['Staatsangehörigkeit (Kategorie) - Total']
Length: 1, dtype: str
<StringArray>
['Geschlecht - Total']
Length: 1, dtype: str


Die Spalten Staatsangehörige und Geschlecht enthalten für alle Zeilen die selben Werte. Für die weitere Analyse sind diese nicht von Bedeutung.

In [121]:
df_population_processed = df_population.drop(columns=["Staatsangehörigkeit (Kategorie)", "Geschlecht"])
print(pd.DataFrame({
        "rows": [len(df_population_processed)],
        "columns": [df_population_processed.shape[1]],
}))

    rows  columns
0  39984        4


Für eine spätere Analyse kann es sinnvoll sein das Alter als einen numerischen Typ zu haben

In [122]:
df_population_processed.rename(columns={
    "Alter": "alter_label",
}, inplace=True)
df_population_processed["alter"] = df_population_processed["alter_label"].str.extract(r"(\d+)").astype("float")
df_population_processed.head()

,Jahr,Kanton,alter_label,Bestand am 31. Dezember,alter
0,2011,Schweiz,Alter - Total,7954662,NaN
1,2011,Schweiz,0 Jahre,78426,0.0
2,2011,Schweiz,1 Jahr,81663,1.0
3,2011,Schweiz,2 Jahre,80196,2.0
4,2011,Schweiz,3 Jahre,79522,3.0


Für eingacheres handling macht es Sinn die Spalten umzubennen.

In [130]:
df_population_processed = df_population_processed.rename(columns={
    "Jahr": "year",
    "Kanton": "canton",
    "alter_label": "age_label",
    "Bestand am 31. Dezember": "population",
    "alter": "age",
})
df_population_processed.head()

,year,canton,age_label,population,age
0,2011,Schweiz,Alter - Total,7954662,NaN
1,2011,Schweiz,0 Jahre,78426,0.0
2,2011,Schweiz,1 Jahr,81663,1.0
3,2011,Schweiz,2 Jahre,80196,2.0
4,2011,Schweiz,3 Jahre,79522,3.0


In [131]:
pd.DataFrame({
        "year_min": [df_population_processed["year"].min()],
        "year_max": [df_population_processed["year"].max()],
        "n_years": [df_population_processed["year"].nunique()],
        "n_cantons": [df_population_processed["canton"].nunique()],
        "n_age_groups": [df_population_processed["age_label"].nunique()],
})

,year_min,year_max,n_years,n_cantons,n_age_groups
0,2011,2024,14,28,102


In [135]:
df_population_ch = df_population_processed[
    (df_population_processed["canton"] == "Schweiz") &
    (df_population_processed["age_label"].eq("Alter - Total"))
].copy()

px.line(
    df_population_ch,
    x="year",
    y="population",
    markers=True,
    title="Bevölkerung Schweiz, Bestand am 31. Dezember",
    labels={"year": "Jahr", "population": "Bestand am 31. Dezember"},
)

In [136]:
df_ch = df_population_processed[df_population_processed["canton"] == "Schweiz"]

official_total = df_ch[df_ch["age_label"] == "Alter - Total"].set_index("year")["population"]

age_sum = (
    df_ch[df_ch["age_label"] != "Alter - Total"]
    .groupby("year")["population"]
    .sum()
)

(age_sum - official_total).abs().max()

np.int64(0)

In [137]:
df_population_age = df_population_processed[df_population_processed["age_label"] != "Alter - Total"].copy()
df_population_age["is_66_plus"] = df_population_age["age"] >= 66

df_population_66_share = (
    df_population_age
    .groupby(["year", "canton", "is_66_plus"], as_index=False)["population"]
    .sum()
)

df_population_66_share["total_population"] = df_population_66_share.groupby(["year", "canton"])["population"].transform("sum")
df_population_66_share["population_share_66_plus"] = df_population_66_share["population"] / df_population_66_share["total_population"]
df_population_66_share = df_population_66_share[df_population_66_share["is_66_plus"]].copy()

df_population_66_share.head()

,year,canton,is_66_plus,population,total_population,population_share_66_plus
1,2011,Aargau,True,91306,618298,0.147673
3,2011,Appenzell Ausserrhoden,True,8947,53313,0.167820
5,2011,Appenzell Innerrhoden,True,2525,15743,0.160389
7,2011,Basel-Landschaft,True,51472,275360,0.186926
9,2011,Basel-Stadt,True,36629,186255,0.196660


In [138]:
px.line(
    df_population_66_share[df_population_66_share["canton"] == "Schweiz"],
    x="year",
    y="population_share_66_plus",
    markers=True,
    title="Anteil 66+ an der Schweizer Bevölkerung",
    labels={"year": "Jahr", "population_share_66_plus": "Anteil 66+"},
)

In [139]:
latest_population_year = df_population_66_share["year"].max()

df_population_66_latest = (
    df_population_66_share[
        (df_population_66_share["year"] == latest_population_year) &
        (~df_population_66_share["canton"].isin(["Schweiz"]))
    ]
    .sort_values("population_share_66_plus", ascending=False)
)

px.bar(
    df_population_66_latest.head(10),
    x="canton",
    y="population_share_66_plus",
    title=f"Top 10 Kantone nach 66+-Anteil {latest_population_year}",
    labels={"canton": "Kanton", "population_share_66_plus": "Anteil 66+"},
)

In [143]:
df_population_processed.to_csv(PROCESSED_DIR / "bevoelkerung.csv", index=False)